In [ ]:
# CYR-GPU-005 - CELL 0: BOOTSTRAP / VERIFY / CALIBRATE
# Thin operator wrapper: reads the preregistration from commit B, copies it
# OUTSIDE the repo, checks out the frozen executable (commit A), verifies
# every hash, calibrates hardware, and resolves the campaign. No science
# logic lives in this notebook (section 41).
import json, subprocess, sys
from pathlib import Path

REPO = Path("/content/An-Ra-the-new-AGI")
BRANCH = "cymek-500m-readiness"
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git",
                    str(REPO)], check=True)
sys.path.insert(0, str(REPO))
import os
os.chdir(REPO)

PREREG_PATH = Path("docs/cymek/experiments/CYR-GPU-005/PREREGISTRATION.json")
PREREG = json.loads(PREREG_PATH.read_text("utf-8"))
EXECUTABLE_SHA = PREREG["executable_sha256"]
# 2. copy the preregistration OUTSIDE the repo before checkout
EXTERNAL = Path("/content/CYR-GPU-005-PREREGISTRATION.json")
EXTERNAL.write_text(json.dumps(PREREG, indent=2), encoding="utf-8")
# 3-4. checkout the frozen executable and assert HEAD
subprocess.run(["git", "checkout", "-q", EXECUTABLE_SHA], check=True)
HEAD = subprocess.run(["git", "rev-parse", "HEAD"], check=True,
                      capture_output=True, text=True).stdout.strip()
assert HEAD == EXECUTABLE_SHA, f"HEAD {HEAD} != frozen {EXECUTABLE_SHA}"
# 5. verify executable hashes against the external preregistration copy
import hashlib
file_hashes = {relative: hashlib.sha256((REPO / relative).read_bytes()).hexdigest()
               for relative in PREREG["executable_files"]}
from v5_experiments import cyr_gpu005 as core
core.assert_freeze_contract(preregistration=json.loads(EXTERNAL.read_text()),
                            executable_sha=EXECUTABLE_SHA, head_sha=HEAD,
                            file_hashes=file_hashes)
print("freeze contract verified at", EXECUTABLE_SHA)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CYR-GPU-005 full scientific mode requires Google Colab GPU")
DEVICE = torch.device("cuda")

from anra_v5.cyr_gpu005_run import run_campaign, production_tokenizer, calibrate, resolve_hardware
tokenizer, identity = production_tokenizer(REPO)
print("tokenizer artifact", identity["artifact_sha256"][:16],
      "special IDs", (identity["pad_id"], identity["bos_id"], identity["eos_id"]))
splits = core.render_t2_worlds()
audit = core.commutation_audit(splits)
assert audit["commutation_free"], audit["findings"]
registry = core.proxy_registry(vocab_size=identity["vocabulary_size"])
calibration = calibrate(spec=registry["MIDI"]["spec"], tokenizer=tokenizer,
                        torch=torch, device=DEVICE,
                        special={"pad_id": identity["pad_id"],
                                 "bos_id": identity["bos_id"],
                                 "eos_id": identity["eos_id"]},
                        train_rows=splits["train"])
RESOLVED = resolve_hardware(calibration=calibration, mode="full")
Path("/content").joinpath("CALIBRATION.json").write_text(json.dumps(calibration, indent=2))
Path("/content").joinpath("RESOLVED_PREREGISTRATION.json").write_text(json.dumps(RESOLVED, indent=2))
print("resolved:", RESOLVED["proxy"], "| acquisition",
      RESOLVED["target_actual_tokens_acquisition"], "tokens x", RESOLVED["parents"], "parents")
print("CYR-GPU-005 PREEXECUTION GATE: PASS")


In [ ]:
# CYR-GPU-005 - CELL 1: RUN OR RESUME
# Full scientific mode through the ONE orchestrator. The campaign deadline,
# timebox, checkpoints, and evidence packaging are handled inside it.
import sys
from pathlib import Path
sys.path.insert(0, "/content/An-Ra-the-new-AGI")
from anra_v5.cyr_gpu005_run import run_campaign

OUT = Path("/content/cyr_gpu005_results")
campaign = run_campaign(out=OUT, mode="full",
                        progress=lambda message: print(message, flush=True))
print("campaign status:", campaign["status"],
      "| verdict:", campaign["decision"]["verdict"])


In [ ]:
# CYR-GPU-005 - CELL 2: VERIFY / PACKAGE / DOWNLOAD
import hashlib, json
from pathlib import Path

OUT = Path("/content/cyr_gpu005_results")
receipt = json.loads((OUT / "campaign_receipt.json").read_text("utf-8"))
bundle = Path(receipt["bundle"]["path"])
digest = hashlib.sha256(bundle.read_bytes()).hexdigest()
assert digest == receipt["bundle"]["sha256"], "bundle hash mismatch"
print("bundle verified:", bundle, digest[:16])
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print("not on Colab; bundle available at", bundle)
